In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install ignite

In [4]:
!python3 --version

Python 3.10.12


In [5]:
import torch # we are going to use pytorch instead of numpy because it's much faster.
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.datasets import make_spd_matrix
from scipy.optimize import minimize
from scipy.linalg import toeplitz
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse, r2_score as R2

#from the Penalized Regression with PyTorch.ipynb notebook

#For all the classes and methods in this assignment you will use the PyTorch library. You should use a double precision data type and the device is either "cpu" or "cuda".

In [6]:
dtype = torch.float64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
#personal note look at all of module 3 Variable_Selection_and_Regularization_Introduction.ipynb notebook and penalized reg notebook


# 1. (5 points) Create your own PyTorch class that implements the method of SCAD regularization and variable selection (smoothly clipped absolute deviations) for linear models. Your development should be based on the following references:

https://andrewcharlesjones.github.io/journal/scad.html
https://www.jstor.org/stable/27640214?seq=1



###I created my PyTorch class that implements SCAD mainly based on Andy Jones' page shown below


In [8]:
def scad_penalty(beta_hat, lambda_val, a_val):
    is_linear = (np.abs(beta_hat) <= lambda_val)
    is_quadratic = np.logical_and(lambda_val < np.abs(beta_hat), np.abs(beta_hat) <= a_val * lambda_val)
    is_constant = (a_val * lambda_val) < np.abs(beta_hat)

    linear_part = lambda_val * np.abs(beta_hat) * is_linear
    quadratic_part = (2 * a_val * lambda_val * np.abs(beta_hat) - beta_hat**2 - lambda_val**2) / (2 * (a_val - 1)) * is_quadratic
    constant_part = (lambda_val**2 * (a + 1)) / 2 * is_constant
    return linear_part + quadratic_part + constant_part

def scad_derivative(beta_hat, lambda_val, a_val):
    return lambda_val * ((beta_hat <= lambda_val) + (a_val * lambda_val - beta_hat)*((a_val * lambda_val - beta_hat) > 0) / ((a_val - 1) * lambda_val) * (beta_hat > lambda_val))

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class SCADRegression(nn.Module):
    def __init__(self, n_features, a=3.7, lambda_=1.0, device='cpu'):
        super(SCADRegression, self).__init__()
        self.linear = nn.Linear(n_features, 1)
        self.a = a
        self.lambda_ = lambda_
        self.device = device

    def forward(self, x):
        return self.linear(x)

    def scad_penalty(self, beta_hat):
        lambda_val = self.lambda_
        a_val = self.a
        is_linear = (torch.abs(beta_hat) <= lambda_val)
        is_quadratic = (lambda_val < torch.abs(beta_hat)) & (torch.abs(beta_hat) <= a_val * lambda_val)
        is_constant = (a_val * lambda_val) < torch.abs(beta_hat)

        linear_part = lambda_val * torch.abs(beta_hat) * is_linear
        quadratic_part = (2 * a_val * lambda_val * torch.abs(beta_hat) - beta_hat**2 - lambda_val**2) / (2 * (a_val - 1)) * is_quadratic
        constant_part = (lambda_val**2 * (a_val + 1)) / 2 * is_constant

        return linear_part + quadratic_part + constant_part

    def regularization(self, beta):
        return torch.sum(self.scad_penalty(beta))

    def fit(self, X, y, lr=1e-6, epochs=100):
        X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)
        self.to(self.device)
        criterion = nn.MSELoss()
        optimizer = optim.SGD(self.parameters(), lr=lr)

        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = self.forward(X_tensor)
            loss = criterion(outputs, y_tensor) + self.regularization(self.linear.weight)
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0:
                print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            predictions = self.forward(X_tensor)
        return predictions.cpu().numpy()

    def get_coefficients(self):
        return self.linear.weight.data.cpu().numpy()



# Test your method one a real data set, and determine a variable selection based on features importance according to SCAD.

In [10]:
df = pd.read_csv('drive/MyDrive/Gelila_WM/DATA_441/Data_Sets/housing.csv')
features = ['crime','industrial','nox','rooms','older','distance','ptratio','lstat']
X = np.array(df[features])
y = np.array(df['cmedv']).reshape(-1,1)
X

array([[6.32000000e-03, 2.30999994e+00, 5.37999988e-01, ...,
        4.09000015e+00, 1.53000002e+01, 4.98000002e+00],
       [2.73100010e-02, 7.07000017e+00, 4.69000012e-01, ...,
        4.96710014e+00, 1.77999992e+01, 9.14000034e+00],
       [2.72900000e-02, 7.07000017e+00, 4.69000012e-01, ...,
        4.96710014e+00, 1.77999992e+01, 4.03000021e+00],
       ...,
       [6.07599990e-02, 1.19300003e+01, 5.73000014e-01, ...,
        2.16750002e+00, 2.10000000e+01, 5.63999987e+00],
       [1.09590001e-01, 1.19300003e+01, 5.73000014e-01, ...,
        2.38890004e+00, 2.10000000e+01, 6.48000002e+00],
       [4.74100000e-02, 1.19300003e+01, 5.73000014e-01, ...,
        2.50500011e+00, 2.10000000e+01, 7.88000011e+00]])

In [11]:
y

array([[24.        ],
       [21.6000004 ],
       [34.7000008 ],
       [33.4000015 ],
       [36.2000008 ],
       [28.7000008 ],
       [22.8999996 ],
       [22.1000004 ],
       [16.5       ],
       [18.8999996 ],
       [15.        ],
       [18.8999996 ],
       [21.7000008 ],
       [20.3999996 ],
       [18.2000008 ],
       [19.8999996 ],
       [23.1000004 ],
       [17.5       ],
       [20.2000008 ],
       [18.2000008 ],
       [13.6000004 ],
       [19.6000004 ],
       [15.1999998 ],
       [14.5       ],
       [15.6000004 ],
       [13.8999996 ],
       [16.6000004 ],
       [14.8000002 ],
       [18.3999996 ],
       [21.        ],
       [12.6999998 ],
       [14.5       ],
       [13.1999998 ],
       [13.1000004 ],
       [13.5       ],
       [18.8999996 ],
       [20.        ],
       [21.        ],
       [24.2000008 ],
       [30.7999992 ],
       [34.9000015 ],
       [26.6000004 ],
       [25.2999992 ],
       [24.7000008 ],
       [21.2000008 ],
       [19

In [12]:
X_tensor = torch.tensor(X, dtype=torch.float64, device = device)
y_tensor = torch.tensor(y, dtype=torch.float64, device = device)

In [13]:
type(y_tensor)

torch.Tensor

In [14]:
n_features = X_tensor.shape[1]
model = SCADRegression(n_features=X.shape[1], device=device)
#model.double()
#y_tensor = y_tensor.double()
model.fit(X_tensor, y_tensor)

<ipython-input-9-c097518225b2>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
<ipython-input-9-c097518225b2>:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)


Epoch [1/100], Loss: 1589.8712
Epoch [11/100], Loss: 1313.1104
Epoch [21/100], Loss: 1097.1718
Epoch [31/100], Loss: 928.6548
Epoch [41/100], Loss: 797.1122
Epoch [51/100], Loss: 694.3978
Epoch [61/100], Loss: 614.1932
Epoch [71/100], Loss: 551.5662
Epoch [81/100], Loss: 502.5851
Epoch [91/100], Loss: 464.2437


In [15]:
import numpy as np
from sklearn.datasets import make_regression

import numpy as np

learned_weights = model.linear.weight.detach().cpu().numpy().flatten()
feature_importances = np.abs(learned_weights)
selected_indices = np.where(feature_importances > 0)[0]
selected_features = [features[i] for i in selected_indices]
for feature in selected_features:
    print(feature)
print("These above are the most important features based on my scad regression model.")



crime
industrial
nox
rooms
older
distance
ptratio
lstat
These above are the most important features based on my scad regression model.


Based on the outcome it appeared that all the features in my dataset contribute significantly to the target variable, which in this case is the 'cmedv' column. It also may be the case where there was overfitting in my model but am unable to detect this.

# 2. (4 points) Based on the simulation design explained in class, generate 500 data sets where the input features have a strong correlation structure (you may consider a 0.9) and apply ElasticNet, SqrtLasso and SCAD to check which method produces the best approximation of an ideal solution, such as a "betastar" you design with a sparsity pattern.

In [16]:
#BELOW IS THE ELASTIC NET CLASS FROM VARIABLE SELECTION NOTEOOK:

class ElasticNet(nn.Module):
    def __init__(self, input_size, alpha=1.0, l1_ratio=0.5):
        """
        Initialize the ElasticNet regression model.

        Args:
            input_size (int): Number of input features.
            alpha (float): Regularization strength. Higher values of alpha
                emphasize L1 regularization, while lower values emphasize L2 regularization.
            l1_ratio (float): The ratio of L1 regularization to the total
                regularization (L1 + L2). It should be between 0 and 1.

        """
        super(ElasticNet, self).__init__()
        self.input_size = input_size
        self.alpha = alpha
        self.l1_ratio = l1_ratio

        # Define the linear regression layer
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        """
        Forward pass of the ElasticNet model.

        Args:
            x (Tensor): Input data with shape (batch_size, input_size).

        Returns:
            Tensor: Predicted values with shape (batch_size, 1).

        """
        return self.linear(x)

    def loss(self, y_pred, y_true):
        """
        Compute the ElasticNet loss function.

        Args:
            y_pred (Tensor): Predicted values with shape (batch_size, 1).
            y_true (Tensor): True target values with shape (batch_size, 1).

        Returns:
            Tensor: The ElasticNet loss.

        """
        mse_loss = nn.MSELoss()(y_pred, y_true)
        l1_reg = torch.norm(self.linear.weight, p=1)
        l2_reg = torch.norm(self.linear.weight, p=2)

        loss = mse_loss + self.alpha * (
            self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg
        )

        return loss

    def fit(self, X, y, num_epochs=100, learning_rate=0.01):
        """
        Fit the ElasticNet model to the training data.

        Args:
            X (Tensor): Input data with shape (num_samples, input_size).
            y (Tensor): Target values with shape (num_samples, 1).
            num_epochs (int): Number of training epochs.
            learning_rate (float): Learning rate for optimization.

        """
        optimizer = optim.SGD(self.parameters(), lr=learning_rate)

        for epoch in range(num_epochs):
            self.train()
            optimizer.zero_grad()
            y_pred = self(X)
            loss = self.loss(y_pred, y)
            loss.backward()
            optimizer.step()

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}")

    def predict(self, X):
        """
        Predict target values for input data.

        Args:
            X (Tensor): Input data with shape (num_samples, input_size).

        Returns:
            Tensor: Predicted values with shape (num_samples, 1).

        """
        self.eval()
        with torch.no_grad():
            y_pred = self(X)
        return y_pred
    def get_coefficients(self):
        """
        Get the coefficients (weights) of the linear regression layer.

        Returns:
            Tensor: Coefficients with shape (output_size, input_size).

        """
        return self.linear.weight

#SQRT LASSO CLASS

# we can call this version PED_Adam because we use the adaptive momentum gradient descent for optimization
class sqrtLasso(nn.Module):
    def __init__(self, input_size, alpha=0.1):
        """
        Initialize the  regression model.


        """
        super(sqrtLasso, self).__init__()
        self.input_size = input_size
        self.alpha = alpha


        # Define the linear regression layer
        self.linear = nn.Linear(input_size, 1).double()

    def forward(self, x):
        """
        Forward pass of the model.

        Args:
            x (Tensor): Input data with shape (batch_size, input_size).

        Returns:
            Tensor: Predicted values with shape (batch_size, 1).

        """
        # Cast the input tensor to double to match the weight tensor data type
        x = x.double()
        return self.linear(x)

    def loss(self, y_pred, y_true):
        """
        Compute the loss function.

        Args:
            y_pred (Tensor): Predicted values with shape (batch_size, 1).
            y_true (Tensor): True target values with shape (batch_size, 1).

        Returns:
            Tensor: The loss.

        """
        mse_loss = nn.MSELoss()(y_pred, y_true)
        l1_reg = torch
#SCAD CLASS:
class SCADRegression(nn.Module):
    def __init__(self, n_features, a=3.7, lambda_=1.0, device='cpu'):
        super(SCADRegression, self).__init__()
        self.linear = nn.Linear(n_features, 1)
        self.a = a
        self.lambda_ = lambda_
        self.device = device

    def forward(self, x):
        return self.linear(x)

    def scad_penalty(self, beta_hat):
        lambda_val = self.lambda_
        a_val = self.a
        is_linear = (torch.abs(beta_hat) <= lambda_val)
        is_quadratic = (lambda_val < torch.abs(beta_hat)) & (torch.abs(beta_hat) <= a_val * lambda_val)
        is_constant = (a_val * lambda_val) < torch.abs(beta_hat)

        linear_part = lambda_val * torch.abs(beta_hat) * is_linear
        quadratic_part = (2 * a_val * lambda_val * torch.abs(beta_hat) - beta_hat**2 - lambda_val**2) / (2 * (a_val - 1)) * is_quadratic
        constant_part = (lambda_val**2 * (a_val + 1)) / 2 * is_constant

        return linear_part + quadratic_part + constant_part

    def regularization(self, beta):
        return torch.sum(self.scad_penalty(beta))

    def fit(self, X, y, lr=1e-6, epochs=100):
        X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)
        self.to(self.device)
        criterion = nn.MSELoss()
        optimizer = optim.SGD(self.parameters(), lr=lr)

        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = self.forward(X_tensor)
            loss = criterion(outputs, y_tensor) + self.regularization(self.linear.weight)
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0:
                print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            predictions = self.forward(X_tensor)
        return predictions.cpu().numpy()

    def get_coefficients(self):
        return self.linear.weight.data.cpu().numpy()




In [ ]:
import numpy as np
import torch

def generate_fivehund_dataset(num_samples, num_features, correlation, betastar, noise_std=0.1):
    random_data = np.random.randn(num_samples, num_features)
    cov_matrix = np.cov(random_data, rowvar=False)
    cov_matrix = correlation * cov_matrix + (1 - correlation) * np.diag(np.ones(num_features))
    X_data = np.random.multivariate_normal(np.zeros(num_features), cov_matrix, num_samples)
    noise = np.random.normal(0, noise_std, num_samples)
    y_data = X_data.dot(betastar) + noise
    return X_data, y_data

def train_and_eval(model, train_data_X, train_data_y, test_data_X, test_data_y, betastar):
    X_train_tensor = torch.tensor(train_data_X, dtype=torch.float64)
    y_train_tensor = torch.tensor(train_data_y, dtype=torch.float64).view(-1, 1)
    X_test_tensor = torch.tensor(test_data_X, dtype=torch.float64)
    y_test_tensor = torch.tensor(test_data_y, dtype=torch.float64).view(-1, 1)

    if isinstance(model, ElasticNet):
        model.fit(X_train_tensor, y_train_tensor, num_iterations=100)
    elif isinstance(model, sqrtLasso):
        model.fit(X_train_tensor, y_train_tensor, num_iterations=100)
    elif isinstance(model, SCADRegression):
        model.fit(X_train_tensor, y_train_tensor, epochs=100)

    y_pred = model.predict(X_test_tensor)
    mse_loss = torch.nn.functional.mse_loss(y_pred, y_test_tensor).item()
    betastar_tensor = torch.tensor(betastar, dtype=torch.float64)
    coeff_error = torch.sum((betastar_tensor - model.get_coefficients().detach().flatten()) ** 2).item()
    return mse_loss, coeff_error



num_samples = 100
num_features = 20
num_datasets = 500
correlation = 0.9 #AS REQUESTED!!!!!
betastar = np.zeros(num_features)
betastar[:5] = np.random.uniform(1, 3, 5)

results = {
    'ElasticNet': {'mse': [], 'recovery_error': []},
    'SqrtLasso': {'mse': [], 'recovery_error': []},
    'SCAD': {'mse': [], 'recovery_error': []}
}

for _ in range(num_datasets):
    X_data, y_data = generate_fivehund_dataset(num_samples, num_features, correlation, betastar)
    split_index = int(num_samples * 0.8)
    train_data_X, test_data_X = X_data[:split_index], X_data[split_index:]
    train_data_y, test_data_y = y_data[:split_index], y_data[split_index:]

    # ElasticNet Model
    elastic_model = ElasticNet(num_features)
    mse_elastic, recovery_error_elastic = train_and_eval(elastic_model, train_data_X, train_data_y, test_data_X, test_data_y, betastar)
    results['ElasticNet']['mse'].append(mse_elastic)
    results['ElasticNet']['recovery_error'].append(recovery_error_elastic)

    # SqrtLasso Model
    sqrt_lasso_model = sqrtLasso(num_features)
    mse_sqrt_lasso, recovery_error_sqrt_lasso = train_and_eval(sqrt_lasso_model, train_data_X, train_data_y, test_data_X, test_data_y, betastar)
    results['sqrtLasso']['mse'].append(mse_sqrt_lasso)
    results['sqrtLasso']['recovery_error'].append(recovery_error_sqrt_lasso)

    # SCAD Model
    scad_model = SCADRegression(num_features)
    mse_scad, recovery_error_scad = train_and_eval(scad_model, train_data_X, train_data_y, test_data_X, test_data_y, betastar)
    results['SCAD']['mse'].append(mse_scad)
    results['SCAD']['recovery_error'].append(recovery_error_scad)


average_results = {model: {metric: np.mean(values) for metric, values in metrics.items()} for model, metrics in results.items()}
print(average_results)


Streaming output truncated to the last 5000 lines.
Epoch 50: Loss 129.95706176757812
Epoch 60: Loss 129.86134338378906
Epoch 70: Loss 129.7657012939453
Epoch 80: Loss 129.670166015625
Epoch 90: Loss 129.5746612548828
Epoch [10/100], Loss: 9.19581413269043
Epoch [20/100], Loss: 9.100560188293457
Epoch [30/100], Loss: 9.019798278808594
Epoch [40/100], Loss: 8.945869445800781
Epoch [50/100], Loss: 8.878151893615723
Epoch [60/100], Loss: 8.816079139709473
Epoch [70/100], Loss: 8.759135246276855
Epoch [80/100], Loss: 8.706853866577148
Epoch [90/100], Loss: 8.658811569213867
Epoch [100/100], Loss: 8.614625930786133
Epoch [100/200], Loss: 13.899430137872695
Epoch [200/200], Loss: 12.086057376861572
Epoch 0: Loss 139.0550537109375
Epoch 10: Loss 138.94439697265625
Epoch 20: Loss 138.83384704589844
Epoch 30: Loss 138.723388671875
Epoch 40: Loss 138.61302185058594
Epoch 50: Loss 138.5027618408203
Epoch 60: Loss 138.392578125
Epoch 70: Loss 138.282470703125
Epoch 80: Loss 138.17247009277344
Epoch

According to the end, where {'ElasticNet': {'mse': 1.676343714952469, 'recovery_error': 15.23274946975708}, 'SqrtLasso': {'mse': 1.5584893686771393, 'recovery_error': 15.331877820968629}, 'SCAD': {'mse': 129.84283177566527, 'recovery_error': 28.568074069976806}}, the model with the lowest MSE is sqrtLasso so I would consider it the best. the one with the lowest recovory is ElasticNet